In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

import sys; import os; sys.path.append(os.path.abspath('..'))
from config import OUTPUTS_FIGURES_DIR

def run_eda(df):
    if df.empty:
        print('Empty dataframe, skipping EDA')
        return

    os.makedirs(OUTPUTS_FIGURES_DIR, exist_ok=True)
    
    # 1. AIS signal density map
    plt.figure(figsize=(10, 6))
    plt.hexbin(df['LON'], df['LAT'], gridsize=100, cmap='inferno', bins='log')
    plt.colorbar(label='log10(count)')
    plt.title('AIS Signal Density Map')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.savefig(os.path.join(OUTPUTS_FIGURES_DIR, 'eda_vessel_density_map.png'), dpi=300, bbox_inches='tight')
    plt.close()

    # 2. SOG distribution
    plt.figure(figsize=(10, 6))
    # We will use top 5 vessel types for clarity
    top_types = df['VesselType'].value_counts().nlargest(5).index
    df_top_types = df[df['VesselType'].isin(top_types)]
    
    sns.histplot(data=df_top_types, x='SOG_kmh', hue='VesselType', multiple='stack', bins=50)
    
    # Annotate mean SOG
    means = df_top_types.groupby('VesselType', observed=False)['SOG_kmh'].mean()
    for i, (vtype, mean_sog) in enumerate(means.items()):
        plt.axvline(mean_sog, color=sns.color_palette()[i], linestyle='dashed', linewidth=2)
        plt.text(mean_sog + 0.5, plt.ylim()[1] * 0.9 - (i * plt.ylim()[1] * 0.05), f'{vtype}: {mean_sog:.1f}', color=sns.color_palette()[i])
    
    plt.title('SOG Distribution by Vessel Type')
    plt.xlabel('Speed Over Ground (km/h)')
    plt.savefig(os.path.join(OUTPUTS_FIGURES_DIR, 'eda_sog_distribution.png'), dpi=300, bbox_inches='tight')
    plt.close()

    # 3. Temporal traffic patterns
    plt.figure(figsize=(12, 6))
    pivot = df.pivot_table(index='day_of_week', columns='hour_of_day', values='MMSI', aggfunc='count')
    if not pivot.empty:
        pivot = pivot.fillna(0).astype(float)
        sns.heatmap(pivot, cmap='YlGnBu')
    plt.title('Temporal Traffic Patterns (Vessel Count)')
    plt.xlabel('Hour of Day')
    plt.ylabel('Day of Week (0=Mon, 6=Sun)')
    plt.savefig(os.path.join(OUTPUTS_FIGURES_DIR, 'eda_temporal_heatmap.png'), dpi=300, bbox_inches='tight')
    plt.close()

    # 4. ETA_hours distribution
    plt.figure(figsize=(10, 6))
    sns.histplot(df['ETA_hours'], bins=50, log_scale=(False, True))
    
    median_eta = df['ETA_hours'].median()
    p95_eta = df['ETA_hours'].quantile(0.95)
    
    plt.axvline(median_eta, color='red', linestyle='dashed', label=f'Median: {median_eta:.1f}h')
    plt.axvline(p95_eta, color='orange', linestyle='dashed', label=f'95th Pct: {p95_eta:.1f}h')
    
    plt.title('ETA Distribution (Log Scale)')
    plt.xlabel('ETA (hours)')
    plt.ylabel('Count (log scale)')
    plt.legend()
    plt.savefig(os.path.join(OUTPUTS_FIGURES_DIR, 'eda_eta_distribution.png'), dpi=300, bbox_inches='tight')
    plt.close()

    # 5. Draft vs SOG scatter
    plt.figure(figsize=(10, 6))
    # Sample for performance if needed, but let's just plot all or sample
    sample_df = df_top_types.sample(min(10000, len(df_top_types)))
    sns.scatterplot(data=sample_df, x='Draft', y='SOG_kmh', hue='VesselType', alpha=0.5, s=10)
    plt.title('Draft vs SOG by Vessel Type')
    plt.xlabel('Draft (m)')
    plt.ylabel('Speed Over Ground (km/h)')
    plt.savefig(os.path.join(OUTPUTS_FIGURES_DIR, 'eda_draft_vs_sog.png'), dpi=300, bbox_inches='tight')
    plt.close()

    # 6. Micro-kinematic zone comparison
    plt.figure(figsize=(10, 6))
    df['Zone'] = df['is_micro_kinematic_zone'].map({1: 'Inside (<50km)', 0: 'Outside (>50km)'})
    sns.violinplot(data=df, x='Zone', y='ETA_hours', hue='Zone', legend=False)
    plt.title('ETA Distribution: Inside vs Outside Micro-Kinematic Zone')
    plt.ylabel('ETA (hours)')
    plt.savefig(os.path.join(OUTPUTS_FIGURES_DIR, 'eda_mkz_violin.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # Print summary statistics
    print(f"Total unique vessels: {df['MMSI'].nunique()}")
    print(f"Date range: {df['BaseDateTime'].min()} to {df['BaseDateTime'].max()}")
    print(f"Mean ETA_hours: {df['ETA_hours'].mean():.2f}")
    print(f"Median ETA_hours: {df['ETA_hours'].median():.2f}")
    
    mkz_pct = (df['is_micro_kinematic_zone'].sum() / len(df)) * 100
    print(f"% of records in MKZ: {mkz_pct:.2f}%")

if __name__ == "__main__":
    import os; import sys; sys.path.append(os.path.abspath(".."))
    df_path = os.path.join(os.path.abspath(".."), "data", "processed", "ais_features.parquet")
    if os.path.exists(df_path):
        df = pd.read_parquet(df_path)
        run_eda(df)
    else:
        print(f"File not found: {df_path}")


Total unique vessels: 9
Date range: 2024-01-03 21:20:00 to 2026-02-28 00:10:00
Mean ETA_hours: 56.30
Median ETA_hours: 37.17
% of records in MKZ: 9.09%
